In [2]:
# when running on Google colab, otherwise the below cell is skipped
if "google.colab" in str(get_ipython()):
    # install biopython
    ! pip install biopython

In [ ]:
from pathlib import Path

from Bio import Align, SeqIO
from Bio.PDB import MMCIFParser, PPBuilder
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

## Reference `.cif` file
First we work with the reference `.cif` file. This `.cif` file must contain only one chain. A multi-chain `.cif` can for example be split using e.g. [pymol](https://github.com/schrodinger/pymol-open-source).

For this example, we will use a cleaned structure of [Ubiquitin](https://www.rcsb.org/structure/1BT0).

In [2]:
WORKDIR = Path.cwd()

input_cif = WORKDIR / "1BT0_A.cif"
cif_base = Path(input_cif).with_suffix("").name

We read the `.cif` file and extract the sequences from it. 

In [3]:
# Read first CIF
parser = MMCIFParser(QUIET=True)
structure1 = parser.get_structure("s1", input_cif)

ppb = PPBuilder()
cif_sequence = "".join(
    str(pep.get_sequence()) for pep in ppb.build_peptides(next(iter(structure1[0])))
)
cif_sequence

'MLIKVKTLTGKEIEIDIEPTDTIDRIKERVEEKEGIPPVQQRLIYAGKQLADDKTAKDYNIEGGSVLHLVLAL'

Now we save the extracted sequence for bookkeeping:

In [4]:
cif_name = input_cif.name
record = SeqRecord(Seq(cif_sequence), id=f"{cif_name}", description="")
SeqIO.write([record], f"{cif_base}_seqs.fasta", "fasta")

print(f"{cif_name}_seqs.fasta saved!")

1BT0_A.cif_seqs.fasta saved!


## Sequence Alignment
Next, but in the query sequence that you want to use in AlphaFold3. In our example, we use the first 30 residues of the [Ubiquitin sequence](https://www.uniprot.org/uniprot/Q9SHE7) to illustrate the concept.

In [5]:
query_sequence = "TMIKVKTLTGKEIEIDIEPTDTIDRIKERV"

The next cell does a simple alignment between the `cif_sequence` and your `query_sequence`. You can find more information in the [biopython documentation](https://biopython.org/docs/dev/Tutorial/chapter_pairwise.html).

In [6]:
aligner = Align.PairwiseAligner()
alignments = aligner.align(cif_sequence, query_sequence)
alignment = alignments[0]

print(alignment)
print("Score:", alignment.score)

target            0 -MLIKVKTLTGKEIEIDIEPTDTIDRIKERVEEKEGIPPVQQRLIYAGKQLADDKTAKDY
                  0 -|-||||||||||||||||||||||||||||-----------------------------
query             0 TM-IKVKTLTGKEIEIDIEPTDTIDRIKERV-----------------------------

target           59 NIEGGSVLHLVLAL 73
                 60 -------------- 74
query            30 -------------- 30

Score: 29.0


If your alignment is not satisfactory, use other tools like [Kalign](https://www.ebi.ac.uk/jdispatcher/msa/kalign). Uncomment the next cells by removing the # and use your own `.aln` file.

In [7]:
# from Bio import AlignIO
# align = AlignIO.read("output.aln", "clustal")

Now we have the alignment and can change it to the format that AlphaFold3 expects.

In [8]:
coords = alignment.coordinates
t, q = coords[0], coords[1]

query_indices, template_indices = [], []

for i in range(len(q) - 1):
    q0, q1 = q[i], q[i + 1]
    t0, t1 = t[i], t[i + 1]
    dq = q1 - q0
    dt = t1 - t0

    if dq > 0 and dt > 0:
        n = min(dq, dt)
        query_indices.extend(range(q0, q0 + n))
        template_indices.extend(range(t0, t0 + n))

print("queryIndices:", query_indices)
print("templateIndices:", template_indices)

queryIndices: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
templateIndices: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


If you run a local instance of AlphaFold3, either on your computer or on a HPC system, copy the two strings into the template section, as explained in the [AlphaFold3 documentation](https://github.com/google-deepmind/alphafold3/blob/main/docs/input.md#structural-templates) together with the `.cif` file that you used as an input above.

If you want to work on AlphaFold3 server, execute the next cell to create the mapping file in `.json` format.

In [9]:
alignment_file_name = f"{cif_base}_alignment.json"

alignment_json = f"""
[
  {{
    "name": "{input_cif}",
    "queryIndices": {list(query_indices)},
    "templateIndices": {list(template_indices)}
  }}
]
"""

with open(alignment_file_name, "w") as f:
    f.write(alignment_json)